# Notebook 18h: Positive Anomaly Detection with DWPC Comparison

## Purpose
Detect POSITIVE anomalies (enrichment) in compound-pathway pairs using
degree-conditioned null model from notebook 18f and variance estimates from
notebook 18g. Focus exclusively on pairs with MORE paths than expected
(biological enrichment).

## Method
1. Load trained model and variance estimates
2. Compute expected pathway counts (model predictions)
3. Compute observed pathway counts (Hetionet)
4. Filter for POSITIVE anomalies only (observed > expected)
5. Calculate Z-scores: z = (observed - expected) / std_from_permutations
6. Compute DWPC scores for enriched pairs
7. Compare anomaly scores to DWPC
8. Identify novel discoveries (high anomaly, low DWPC)

## Key Insight
We focus ONLY on positive anomalies (enrichment, not depletion) because:
- Positive anomalies indicate biological signal beyond random degree structure
- DWPC captures degree-weighted connectivity
- Novel discoveries = high Z-score + low DWPC (not explained by degree weighting)

## Inputs
- `results/pathway_nn/trained_models/{metapath}_Degree_Sig_NN.pt`
- `results/pathway_nn/variance_analysis/{metapath}_variance_estimates.csv`
- `data/edges/{edge_type}.sparse.npz` (original Hetionet)

## Outputs
- `results/pathway_nn/anomaly_detection/{metapath}_all_anomalies.csv`
- `results/pathway_nn/anomaly_detection/{metapath}_significant_anomalies.csv`
- `results/pathway_nn/anomaly_detection/{metapath}_novel_discoveries.csv`
- `results/pathway_nn/anomaly_detection/{metapath}_anomaly_summary.json`
- `results/pathway_nn/anomaly_detection/{metapath}_volcano_plot.png`
- `results/pathway_nn/anomaly_detection/{metapath}_dwpc_comparison.png`

## Usage
```bash
# Local execution
jupyter nbconvert --execute notebooks/18h_anomaly_detection.ipynb

# HPC execution with papermill
papermill notebooks/18h_anomaly_detection.ipynb \
    notebooks/executed/18h_anomaly_detection_executed.ipynb \
    -p metapath "CbGpPW"
```

## References
- Himmelstein et al. (2017). eLife. https://doi.org/10.7554/eLife.26726
  - DWPC (Degree-Weighted Path Count) method

In [ ]:
# Papermill parameters
metapath = 'CbGpPW'
edge1_type = 'CbG'
edge2_type = 'GpPW'
significance_threshold = 0.01  # P-value threshold
min_pathway_count = 1  # Minimum paths to consider
n_degree_bins = 10
n_inter_bins = 10
dwpc_damping = 0.4  # DWPC damping exponent
random_seed = 42

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from scipy.stats import norm, pearsonr, spearmanr
from matplotlib.patches import Patch
import sys

repo_dir = Path.cwd().parent
sys.path.insert(0, str(repo_dir))

from src.models.degree_signature_nn import DegreeSignatureNN
from src.intermediate_signatures import (
    compute_intermediate_signature,
    create_degree_bins,
    assign_to_bins
)
from src.node_labels import load_node_labels

print(f"Anomaly detection for {metapath}")
print(f"Significance threshold: p < {significance_threshold}")

In [ ]:
# Load trained model
model_file = (repo_dir / 'results' / 'pathway_nn' / 'trained_models' /
              f'{metapath}_Degree_Sig_NN.pt')
if not model_file.exists():
    raise FileNotFoundError(
        f"Trained model not found: {model_file}\n"
        "Please run notebook 18f first!"
    )

model = DegreeSignatureNN.load(model_file)
print(f"✓ Loaded model: {model_file}")

# Load variance estimates from notebook 18g
variance_file = (repo_dir / 'results' / 'pathway_nn' / 'variance_analysis' /
                 f'{metapath}_variance_estimates.csv')
if not variance_file.exists():
    raise FileNotFoundError(
        f"Variance estimates not found: {variance_file}\n"
        "Please run notebook 18g first!"
    )

variance_df = pd.read_csv(variance_file)
print(f"✓ Loaded variance estimates: {variance_file}")
print(f"  Variance estimates for {len(variance_df)} bin combinations")

In [ ]:
# Load original Hetionet edges
data_dir = repo_dir / 'data'
edge1_file = data_dir / 'edges' / f'{edge1_type}.sparse.npz'
if not edge1_file.exists():
    edge1_file = (data_dir / 'permutations' / '000.hetmat' / 'edges' /
                  f'{edge1_type}.sparse.npz')

edge2_file = data_dir / 'edges' / f'{edge2_type}.sparse.npz'
if not edge2_file.exists():
    edge2_file = (data_dir / 'permutations' / '000.hetmat' / 'edges' /
                  f'{edge2_type}.sparse.npz')

if not edge1_file.exists() or not edge2_file.exists():
    raise FileNotFoundError("Edge files not found")

edge1_matrix = sp.load_npz(str(edge1_file))
edge2_matrix = sp.load_npz(str(edge2_file))

print(f"✓ Loaded edge matrices:")
print(f"  Edge1 ({edge1_type}): {edge1_matrix.shape}")
print(f"  Edge2 ({edge2_type}): {edge2_matrix.shape}")

# CRITICAL: Convert boolean matrices to numeric BEFORE computing pathways
# Boolean @ Boolean gives reachability (True/False), NOT path counts!
if edge1_matrix.dtype == bool or edge1_matrix.dtype == np.bool_:
    print(f"Converting Edge1 from boolean to int32 for path counting")
    edge1_matrix = edge1_matrix.astype(np.int32)
if edge2_matrix.dtype == bool or edge2_matrix.dtype == np.bool_:
    print(f"Converting Edge2 from boolean to int32 for path counting")
    edge2_matrix = edge2_matrix.astype(np.int32)

print(f"After conversion:")
print(f"  Edge1 dtype: {edge1_matrix.dtype}")
print(f"  Edge2 dtype: {edge2_matrix.dtype}")

# Compute pathway counts (edge matrices already converted to int32)
pathway_matrix = edge1_matrix @ edge2_matrix

print(f"  Pathway matrix: {pathway_matrix.shape}, "
      f"{pathway_matrix.nnz:,} non-zero paths")
print(f"  Pathway matrix dtype: {pathway_matrix.dtype}")
print(f"  Value range: {pathway_matrix.data.min()} - {pathway_matrix.data.max()}")

In [ ]:
# Compute degrees
source_degrees = np.asarray(edge1_matrix.sum(axis=1)).ravel()
target_degrees = np.asarray(edge2_matrix.sum(axis=1)).ravel()

# Create degree bins
source_bins = create_degree_bins(source_degrees, n_degree_bins)
target_bins = create_degree_bins(target_degrees, n_degree_bins)

# Assign nodes to bins
source_bin_assignments = assign_to_bins(source_degrees, source_bins)
target_bin_assignments = assign_to_bins(target_degrees, target_bins)

print(f"✓ Created degree bins:")
print(f"  Source bins: {len(source_bins)-1} bins")
print(f"  Target bins: {len(target_bins)-1} bins")

In [ ]:
# Compute anomaly scores for POSITIVE anomalies only
# CRITICAL: We ONLY consider pairs where actual_count > expected_count
# (biological enrichment, not depletion)
print(f"\nComputing POSITIVE anomaly scores (enrichment only)...")

pathway_coo = pathway_matrix.tocoo()
anomaly_scores = []
n_positive_filtered = 0
n_total_processed = 0

for idx, (src_idx, tgt_idx, actual_count) in enumerate(
    zip(pathway_coo.row, pathway_coo.col, pathway_coo.data)):
    
    if idx % 1000 == 0:
        print(f"  Processed {idx:,}/{pathway_coo.nnz:,} pairs...", end='\r')
    
    n_total_processed += 1
    
    # Skip if below minimum threshold
    if actual_count < min_pathway_count:
        continue
    
    # Get bin assignments
    src_bin = source_bin_assignments[src_idx]
    tgt_bin = target_bin_assignments[tgt_idx]
    
    # Compute intermediate signature for this specific pair
    intermediate_nodes = edge1_matrix[src_idx].nonzero()[1]
    
    if len(intermediate_nodes) == 0:
        continue
    
    # Get degrees of intermediate nodes
    intermediate_degrees_in = np.array([
        edge1_matrix[:, g].sum() for g in intermediate_nodes
    ])
    intermediate_degrees_out = np.array([
        edge2_matrix[g, :].sum() for g in intermediate_nodes
    ])
    
    # Create intermediate signature (10x10 histogram)
    inter_sig = np.zeros((n_inter_bins, n_inter_bins))
    if len(intermediate_nodes) > 0:
        in_bins_edges = np.percentile(
            intermediate_degrees_in,
            np.linspace(0, 100, n_inter_bins+1)
        )
        out_bins_edges = np.percentile(
            intermediate_degrees_out,
            np.linspace(0, 100, n_inter_bins+1)
        )
        
        in_bins_idx = np.digitize(intermediate_degrees_in, in_bins_edges)
        out_bins_idx = np.digitize(intermediate_degrees_out, out_bins_edges)
        
        for in_b, out_b in zip(in_bins_idx, out_bins_idx):
            in_b = min(max(in_b - 1, 0), n_inter_bins - 1)
            out_b = min(max(out_b - 1, 0), n_inter_bins - 1)
            inter_sig[in_b, out_b] += 1
        
        # Normalize
        if inter_sig.sum() > 0:
            inter_sig = inter_sig / inter_sig.sum()
    
    inter_sig_flat = inter_sig.flatten()
    
    # Create feature vector
    X = np.array([[src_bin, tgt_bin, *inter_sig_flat]], dtype=np.float32)
    
    # Get model prediction (expected count given degree structure)
    expected_count = model.predict(X)[0]
    
    # CRITICAL FILTER: Only keep POSITIVE anomalies (enrichment)
    # Skip if actual <= expected (no enrichment)
    if actual_count <= expected_count:
        continue
    
    n_positive_filtered += 1
    
    # Get variance estimate for this bin combination
    variance_row = variance_df[
        (variance_df['source_bin'] == src_bin) &
        (variance_df['target_bin'] == tgt_bin)
    ]
    
    if len(variance_row) > 0:
        expected_std = variance_row['std_count_across_perms'].values[0]
    else:
        # Fallback: use mean std if bin combo not found
        expected_std = variance_df['std_count_across_perms'].mean()
    
    # Compute Z-score (anomaly score)
    # All Z-scores will be positive since actual_count > expected_count
    if expected_std > 0:
        z_score = (actual_count - expected_count) / expected_std
    else:
        z_score = 0.0
    
    # Compute p-value (one-tailed test for enrichment)
    p_value = 1 - norm.cdf(z_score)
    
    anomaly_scores.append({
        'source_idx': src_idx,
        'target_idx': tgt_idx,
        'source_degree': source_degrees[src_idx],
        'target_degree': target_degrees[tgt_idx],
        'source_bin': src_bin,
        'target_bin': tgt_bin,
        'actual_count': actual_count,
        'expected_count': expected_count,
        'expected_std': expected_std,
        'z_score': z_score,
        'p_value': p_value
    })

print(f"\n✓ Positive anomaly analysis complete:")
print(f"  Total pairs processed: {n_total_processed:,}")
print(f"  Positive anomalies (actual > expected): {n_positive_filtered:,}")
print(f"  Retention rate: {100*n_positive_filtered/n_total_processed:.2f}%")

In [ ]:
# Create anomaly dataframe (all positive by construction)
anomaly_df = pd.DataFrame(anomaly_scores)

# Apply multiple testing correction (Bonferroni)
n_tests = len(anomaly_df)
bonferroni_threshold = significance_threshold / n_tests
anomaly_df['significant'] = anomaly_df['p_value'] < significance_threshold
anomaly_df['significant_bonferroni'] = (
    anomaly_df['p_value'] < bonferroni_threshold
)

print(f"\n{'='*70}")
print(f"POSITIVE ANOMALY DETECTION RESULTS (Enrichment Only)")
print(f"{'='*70}")
print(f"\nTotal enriched pairs analyzed: {len(anomaly_df):,}")
print(f"  (pairs where actual > expected)")
print(f"\nSignificant enrichments (p < {significance_threshold}): "
      f"{anomaly_df['significant'].sum():,}")
print(f"Bonferroni-corrected (p < {bonferroni_threshold:.2e}): "
      f"{anomaly_df['significant_bonferroni'].sum():,}")
print(f"\nZ-score range: [{anomaly_df['z_score'].min():.2f}, "
      f"{anomaly_df['z_score'].max():.2f}]")
print(f"Mean Z-score: {anomaly_df['z_score'].mean():.2f}")

In [ ]:
# Load node labels for interpretability
print(f"\nLoading node labels...")

# Determine source and target node types from edge types
# Edge type format: NodeType1-relationship-NodeType2  (e.g., CbG = Compound binds Gene)
source_node_type = 'Compound'  # For metapath CbGpPW, source is Compound
target_node_type = 'Pathway'   # Target is Pathway

try:
    source_labels = load_node_labels(data_dir, source_node_type)
    target_labels = load_node_labels(data_dir, target_node_type)
    
    # Add labels to dataframe
    anomaly_df['source_name'] = anomaly_df['source_idx'].apply(
        lambda idx: source_labels[int(idx)] if idx < len(source_labels) else f'Unknown_{idx}'
    )
    anomaly_df['target_name'] = anomaly_df['target_idx'].apply(
        lambda idx: target_labels[int(idx)] if idx < len(target_labels) else f'Unknown_{idx}'
    )
    
    print(f"✓ Mapped {len(source_labels)} {source_node_type}s and {len(target_labels)} {target_node_type}s")
    
except Exception as e:
    print(f"⚠️  Warning: Could not load node labels: {e}")
    print("   Continuing with indices only")
    anomaly_df['source_name'] = 'Compound_' + anomaly_df['source_idx'].astype(str)
    anomaly_df['target_name'] = 'Pathway_' + anomaly_df['target_idx'].astype(str)

In [ ]:
# Sort by z-score (descending - highest enrichment first)
anomaly_df_sorted = anomaly_df.sort_values('z_score', ascending=False)

print(f"\n{'='*70}")
print(f"TOP 20 ENRICHED PAIRS (Highest Z-scores)")
print(f"{'='*70}")

# Display with human-readable names
display_cols = [
    'source_name', 'target_name', 'source_degree', 'target_degree',
    'actual_count', 'expected_count', 'z_score', 'p_value'
]
print(anomaly_df_sorted[display_cols].head(20).to_string(index=False))

In [ ]:
# Compute DWPC scores
print(f"\nComputing DWPC scores with damping exponent {dwpc_damping}...")

# DWPC formula: DWPC = sum over paths of product of edge weights
# Edge weight = 1 / (degree ^ damping)

# Compute degree-weighted matrices
edge1_degrees = np.asarray(edge1_matrix.sum(axis=1)).ravel()
edge2_degrees_out = np.asarray(edge2_matrix.sum(axis=1)).ravel()

# Create weighted matrices
edge1_weighted = edge1_matrix.copy().tocsr().astype(np.float64)
edge2_weighted = edge2_matrix.copy().tocsr().astype(np.float64)

# Apply degree weighting
for i in range(edge1_weighted.shape[0]):
    if edge1_degrees[i] > 0:
        edge1_weighted.data[edge1_weighted.indptr[i]:edge1_weighted.indptr[i+1]] /= (
            edge1_degrees[i] ** dwpc_damping
        )

for i in range(edge2_weighted.shape[0]):
    if edge2_degrees_out[i] > 0:
        edge2_weighted.data[edge2_weighted.indptr[i]:edge2_weighted.indptr[i+1]] /= (
            edge2_degrees_out[i] ** dwpc_damping
        )

# Compute DWPC matrix
dwpc_matrix = edge1_weighted @ edge2_weighted

print(f"✓ DWPC computed: {dwpc_matrix.shape}")

# Add DWPC to anomaly dataframe
anomaly_df['dwpc_score'] = anomaly_df.apply(
    lambda row: dwpc_matrix[int(row['source_idx']), int(row['target_idx'])],
    axis=1
)

In [ ]:
# Analyze correlation between Z-score and DWPC (for enriched pairs)
pearson_r, pearson_p = pearsonr(anomaly_df['z_score'], anomaly_df['dwpc_score'])
spearman_r, spearman_p = spearmanr(
    anomaly_df['z_score'],
    anomaly_df['dwpc_score']
)

print(f"\n{'='*70}")
print(f"DWPC vs ANOMALY SCORE COMPARISON (Enriched Pairs Only)")
print(f"{'='*70}")
print(f"Pearson correlation:  r = {pearson_r:.4f} (p = {pearson_p:.2e})")
print(f"Spearman correlation: ρ = {spearman_r:.4f} (p = {spearman_p:.2e})")

# Quadrant analysis (75th percentile thresholds)
high_dwpc_threshold = anomaly_df['dwpc_score'].quantile(0.75)
high_z_threshold = anomaly_df['z_score'].quantile(0.75)

quadrants = {
    'high_dwpc_high_z': (
        (anomaly_df['dwpc_score'] >= high_dwpc_threshold) &
        (anomaly_df['z_score'] >= high_z_threshold)
    ).sum(),
    'high_dwpc_low_z': (
        (anomaly_df['dwpc_score'] >= high_dwpc_threshold) &
        (anomaly_df['z_score'] < high_z_threshold)
    ).sum(),
    'low_dwpc_high_z': (
        (anomaly_df['dwpc_score'] < high_dwpc_threshold) &
        (anomaly_df['z_score'] >= high_z_threshold)
    ).sum(),
    'low_dwpc_low_z': (
        (anomaly_df['dwpc_score'] < high_dwpc_threshold) &
        (anomaly_df['z_score'] < high_z_threshold)
    ).sum()
}

print(f"\nQuadrant Analysis (75th percentile thresholds):")
print(f"  High DWPC + High Z-score: {quadrants['high_dwpc_high_z']:,} "
      f"(validated enrichments)")
print(f"  High DWPC + Lower Z-score: {quadrants['high_dwpc_low_z']:,} "
      f"(degree-driven enrichments)")
print(f"  Lower DWPC + High Z-score: {quadrants['low_dwpc_high_z']:,} "
      f"(novel discoveries - MOST INTERESTING)")
print(f"  Lower DWPC + Lower Z-score: {quadrants['low_dwpc_low_z']:,} "
      f"(modest enrichments)")

# Rank overlap analysis
top_n = 100
top_dwpc_indices = set(anomaly_df.nlargest(top_n, 'dwpc_score').index)
top_z_indices = set(anomaly_df.nlargest(top_n, 'z_score').index)
overlap = len(top_dwpc_indices & top_z_indices)

print(f"\nRank Overlap Analysis (Top {top_n} enriched pairs):")
print(f"  Overlap: {overlap}/{top_n} ({100*overlap/top_n:.1f}%)")
print(f"  Unique to DWPC: {len(top_dwpc_indices - top_z_indices)}")
print(f"  Unique to Z-score: {len(top_z_indices - top_dwpc_indices)}")
print(f"{'='*70}")

In [ ]:
# Define output directory
output_dir = repo_dir / 'results' / 'pathway_nn' / 'anomaly_detection'
output_dir.mkdir(parents=True, exist_ok=True)

# Identify novel discoveries (Low DWPC + High Z-score)
novel_discoveries = anomaly_df[
    (anomaly_df['dwpc_score'] < high_dwpc_threshold) &
    (anomaly_df['z_score'] >= high_z_threshold) &
    (anomaly_df['significant'])
].sort_values('z_score', ascending=False)

print(f"\n{'='*70}")
print(f"NOVEL DISCOVERIES (Low DWPC + High Anomaly Score)")
print(f"{'='*70}")
print(f"Found {len(novel_discoveries)} novel associations")
print(f"\nTop 20 Novel Discoveries:")

# Display with human-readable names
display_cols = [
    'source_name', 'target_name', 'actual_count',
    'expected_count', 'z_score', 'dwpc_score', 'p_value'
]
print(novel_discoveries[display_cols].head(20).to_string(index=False))

# Save novel discoveries (includes all columns with names)
novel_file = output_dir / f'{metapath}_novel_discoveries.csv'
novel_discoveries.to_csv(novel_file, index=False)
print(f"\n✓ Saved novel discoveries: {novel_file}")

In [ ]:
# DWPC Visualization (Enriched Pairs Only)
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Scatter: DWPC vs Z-score (all enriched pairs are significant by definition)
colors = ['red' if s else 'orange' for s in anomaly_df['significant']]
axes[0, 0].scatter(anomaly_df['dwpc_score'], anomaly_df['z_score'],
                   c=colors, alpha=0.5, s=20, edgecolors='none')
axes[0, 0].set_xlabel('DWPC Score', fontsize=12)
axes[0, 0].set_ylabel('Anomaly Z-score (Enrichment)', fontsize=12)
axes[0, 0].set_title(f'DWPC vs Anomaly Score (Positive Only)\n(r = {pearson_r:.4f})',
                     fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
           markersize=8, label=f'Significant (p<{significance_threshold})'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='orange',
           markersize=8, label='Non-significant')
]
axes[0, 0].legend(handles=legend_elements, loc='best', fontsize=9)

# 2. Quadrant plot
high_dwpc = anomaly_df['dwpc_score'] >= high_dwpc_threshold
high_z = anomaly_df['z_score'] >= high_z_threshold

axes[0, 1].scatter(anomaly_df.loc[~high_dwpc & ~high_z, 'dwpc_score'],
                   anomaly_df.loc[~high_dwpc & ~high_z, 'z_score'],
                   c='lightgray', alpha=0.3, s=10, label='Lower/Lower')
axes[0, 1].scatter(anomaly_df.loc[high_dwpc & ~high_z, 'dwpc_score'],
                   anomaly_df.loc[high_dwpc & ~high_z, 'z_score'],
                   c='blue', alpha=0.6, s=20, label='High DWPC/Lower Z')
axes[0, 1].scatter(anomaly_df.loc[~high_dwpc & high_z, 'dwpc_score'],
                   anomaly_df.loc[~high_dwpc & high_z, 'z_score'],
                   c='orange', alpha=0.6, s=20, label='Lower DWPC/High Z (Novel!)')
axes[0, 1].scatter(anomaly_df.loc[high_dwpc & high_z, 'dwpc_score'],
                   anomaly_df.loc[high_dwpc & high_z, 'z_score'],
                   c='green', alpha=0.6, s=20,
                   label='High DWPC/High Z (Validated)')
axes[0, 1].axhline(high_z_threshold, color='red', linestyle='--', lw=1)
axes[0, 1].axvline(high_dwpc_threshold, color='red', linestyle='--', lw=1)
axes[0, 1].set_xlabel('DWPC Score', fontsize=12)
axes[0, 1].set_ylabel('Anomaly Z-score (Enrichment)', fontsize=12)
axes[0, 1].set_title('Quadrant Analysis (Enriched Pairs)', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='best', fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# 3. Rank comparison
top_100_dwpc = anomaly_df.nlargest(100, 'dwpc_score').copy()
top_100_dwpc['rank_dwpc'] = range(1, 101)
top_100_z = anomaly_df.nlargest(100, 'z_score').copy()
top_100_z['rank_z'] = range(1, 101)

rank_comparison = top_100_dwpc.merge(
    top_100_z, on=['source_idx', 'target_idx'],
    how='outer', suffixes=('_dwpc', '_z')
)

axes[1, 0].scatter(rank_comparison['rank_dwpc'], rank_comparison['rank_z'],
                   alpha=0.6, s=30, edgecolors='k', linewidth=0.5)
axes[1, 0].plot([1, 100], [1, 100], 'r--', lw=2, label='Perfect agreement')
axes[1, 0].set_xlabel('DWPC Rank', fontsize=12)
axes[1, 0].set_ylabel('Z-score Rank', fontsize=12)
axes[1, 0].set_title(f'Rank Comparison (Top 100 Enriched)\nOverlap: {overlap}/100',
                     fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Distribution comparison
axes[1, 1].hist(anomaly_df['dwpc_score'], bins=50, alpha=0.5,
                label='DWPC', color='blue', edgecolor='black')
ax2 = axes[1, 1].twinx()
ax2.hist(anomaly_df['z_score'], bins=50, alpha=0.5,
         label='Z-score', color='red', edgecolor='black')
axes[1, 1].set_xlabel('DWPC Score', fontsize=12, color='blue')
ax2.set_xlabel('Z-score (Enrichment)', fontsize=12, color='red')
axes[1, 1].set_ylabel('Frequency (DWPC)', fontsize=12, color='blue')
ax2.set_ylabel('Frequency (Z-score)', fontsize=12, color='red')
axes[1, 1].set_title('Score Distributions (Positive Anomalies Only)', fontsize=13, fontweight='bold')
axes[1, 1].tick_params(axis='y', labelcolor='blue')
ax2.tick_params(axis='y', labelcolor='red')
axes[1, 1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()

# Save plot
dwpc_plot_file = output_dir / f'{metapath}_dwpc_comparison.png'
plt.savefig(dwpc_plot_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved DWPC comparison plot: {dwpc_plot_file}")

In [ ]:
# Volcano plot: Z-score vs -log10(p-value) for enriched pairs
fig, ax = plt.subplots(figsize=(12, 8))

# Color by significance (all are enrichments, so just red vs orange)
colors = ['red' if s else 'orange' for s in anomaly_df['significant']]

# Compute -log10(p-value)
minus_log10_p = -np.log10(anomaly_df['p_value'].clip(lower=1e-100))

ax.scatter(anomaly_df['z_score'], minus_log10_p, c=colors, alpha=0.6,
           s=30, edgecolors='k', linewidth=0.3)

# Add significance threshold line
sig_line = -np.log10(significance_threshold)
ax.axhline(sig_line, color='blue', linestyle='--', lw=2,
           label=f'p = {significance_threshold}')

# Add labels
ax.set_xlabel('Anomaly Z-score (Enrichment)', fontsize=13)
ax.set_ylabel('-log10(p-value)', fontsize=13)
ax.set_title(f'Volcano Plot: {metapath} (Positive Anomalies Only)\n'
             f'{anomaly_df["significant"].sum():,} significant enrichments',
             fontsize=14, fontweight='bold')

# Add legend
legend_elements = [
    Patch(facecolor='red', edgecolor='k', label='Significant enrichment'),
    Patch(facecolor='orange', edgecolor='k', label='Non-significant enrichment')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
volcano_file = output_dir / f'{metapath}_volcano_plot.png'
plt.savefig(volcano_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved volcano plot: {volcano_file}")

In [ ]:
# Distribution and scatter plots (Enriched Pairs Only)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Z-score distribution (all positive by construction)
axes[0, 0].hist(anomaly_df['z_score'], bins=50, color='steelblue',
                edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Anomaly Z-score (Enrichment)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title(
    f'Z-score Distribution (Positive Only)\nmean = {anomaly_df["z_score"].mean():.3f}, '
    f'std = {anomaly_df["z_score"].std():.3f}',
    fontsize=13, fontweight='bold'
)
axes[0, 0].grid(True, alpha=0.3)

# 2. P-value distribution
axes[0, 1].hist(anomaly_df['p_value'], bins=50, color='orange',
                edgecolor='black', alpha=0.7)
axes[0, 1].axvline(significance_threshold, color='red', linestyle='--',
                   lw=2, label=f'p = {significance_threshold}')
axes[0, 1].set_xlabel('P-value (One-tailed)', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('P-value Distribution', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Actual vs Expected scatter plot
colors_scatter = ['red' if s else 'gray' for s in anomaly_df['significant']]
axes[1, 0].scatter(anomaly_df['expected_count'], anomaly_df['actual_count'],
                   c=colors_scatter, alpha=0.5, s=20, edgecolors='none')
# Add diagonal line (y = x) where actual = expected
max_val = max(anomaly_df['actual_count'].max(), anomaly_df['expected_count'].max())
axes[1, 0].plot([0, max_val], [0, max_val], 'k--', lw=2, label='actual = expected')
axes[1, 0].set_xlabel('Expected Count (Model Prediction)', fontsize=12)
axes[1, 0].set_ylabel('Actual Count (Observed)', fontsize=12)
axes[1, 0].set_title('Actual vs Expected Pathway Counts\n(All points above diagonal)',
                     fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Significance level breakdown
sig_counts = {
    f'p < {significance_threshold}': anomaly_df['significant'].sum(),
    f'p < {bonferroni_threshold:.2e}': (
        anomaly_df['significant_bonferroni'].sum()
    ),
    'All enriched': len(anomaly_df)
}
axes[1, 1].bar(range(len(sig_counts)), list(sig_counts.values()),
               color=['red', 'darkred', 'gray'], edgecolor='black', alpha=0.7)
axes[1, 1].set_xticks(range(len(sig_counts)))
axes[1, 1].set_xticklabels(list(sig_counts.keys()), rotation=15, ha='right')
axes[1, 1].set_ylabel('Count', fontsize=12)
axes[1, 1].set_title('Significance Levels', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, axis='y', alpha=0.3)
axes[1, 1].set_yscale('log')

plt.tight_layout()

# Save plot
dist_file = output_dir / f'{metapath}_anomaly_distributions.png'
plt.savefig(dist_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved distribution plots: {dist_file}")

In [ ]:
# Save all results to CSV files
print(f"\nSaving results...")

# 1. All enriched pairs (positive anomalies)
all_file = output_dir / f'{metapath}_all_anomalies.csv'
anomaly_df.to_csv(all_file, index=False)
print(f"✓ Saved all enriched pairs: {all_file}")

# 2. Significant enrichments (p < 0.01)
sig_file = output_dir / f'{metapath}_significant_anomalies.csv'
anomaly_df[anomaly_df['significant']].to_csv(sig_file, index=False)
print(f"✓ Saved significant enrichments: {sig_file}")

# 3. Significant enrichments at multiple thresholds
for p_thresh in [0.005, 0.001, 0.0001]:
    sig_subset = anomaly_df[anomaly_df['p_value'] < p_thresh]
    thresh_str = str(p_thresh).replace('.', '')
    thresh_file = output_dir / f'{metapath}_anomalies_p{thresh_str}.csv'
    sig_subset.to_csv(thresh_file, index=False)
    print(f"✓ Saved p < {p_thresh} enrichments: {thresh_file}")

# 4. Novel discoveries (already saved above, but confirming)
print(f"✓ Novel discoveries: {novel_file}")

# 5. Save summary JSON
summary = {
    'metapath': metapath,
    'edge1_type': edge1_type,
    'edge2_type': edge2_type,
    'n_source_nodes': int(edge1_matrix.shape[0]),
    'n_target_nodes': int(edge2_matrix.shape[1]),
    'n_intermediate_nodes': int(edge1_matrix.shape[1]),
    'total_enriched_pairs_analyzed': int(len(anomaly_df)),
    'n_significant': int(anomaly_df['significant'].sum()),
    'n_significant_bonferroni': int(anomaly_df['significant_bonferroni'].sum()),
    'n_novel_discoveries': int(len(novel_discoveries)),
    'significance_threshold': significance_threshold,
    'bonferroni_threshold': bonferroni_threshold,
    'dwpc_damping': dwpc_damping,
    'analysis_type': 'positive_anomalies_only',
    'dwpc_vs_zscore_correlation': {
        'pearson_r': float(pearson_r),
        'pearson_p': float(pearson_p),
        'spearman_r': float(spearman_r),
        'spearman_p': float(spearman_p)
    },
    'quadrant_analysis': {
        'high_dwpc_high_z': int(quadrants['high_dwpc_high_z']),
        'high_dwpc_low_z': int(quadrants['high_dwpc_low_z']),
        'low_dwpc_high_z': int(quadrants['low_dwpc_high_z']),
        'low_dwpc_low_z': int(quadrants['low_dwpc_low_z'])
    },
    'rank_overlap_top100': int(overlap),
    'z_score_stats': {
        'mean': float(anomaly_df['z_score'].mean()),
        'std': float(anomaly_df['z_score'].std()),
        'min': float(anomaly_df['z_score'].min()),
        'max': float(anomaly_df['z_score'].max()),
        'q25': float(anomaly_df['z_score'].quantile(0.25)),
        'median': float(anomaly_df['z_score'].median()),
        'q75': float(anomaly_df['z_score'].quantile(0.75))
    },
    'dwpc_score_stats': {
        'mean': float(anomaly_df['dwpc_score'].mean()),
        'std': float(anomaly_df['dwpc_score'].std()),
        'min': float(anomaly_df['dwpc_score'].min()),
        'max': float(anomaly_df['dwpc_score'].max()),
        'q25': float(anomaly_df['dwpc_score'].quantile(0.25)),
        'median': float(anomaly_df['dwpc_score'].median()),
        'q75': float(anomaly_df['dwpc_score'].quantile(0.75))
    }
}

summary_file = output_dir / f'{metapath}_anomaly_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✓ Saved summary: {summary_file}")

## Summary

### Positive Anomaly Detection Completed Successfully

**Analysis Scope:**
- **POSITIVE ANOMALIES ONLY**: Analyzed compound-pathway pairs with MORE paths than expected (biological enrichment)
- Filtered out all negative anomalies (depletion) to focus on biological signal

**Key Results:**
- Analyzed {:,} enriched compound-pathway pairs (actual > expected)
- Identified {:,} significant enrichments (p < {})
- Found {:,} novel discoveries (high anomaly, low DWPC)
- DWPC vs Z-score correlation: r = {:.4f}

**Output Files:**
1. `{}_all_anomalies.csv` - Complete results for all enriched pairs
2. `{}_significant_anomalies.csv` - Filtered by p < {}
3. `{}_novel_discoveries.csv` - Low DWPC + high Z-score
4. `{}_anomaly_summary.json` - Metadata and statistics
5. `{}_volcano_plot.png` - Z-score vs -log10(p-value)
6. `{}_dwpc_comparison.png` - 4-panel DWPC analysis
7. `{}_anomaly_distributions.png` - Distribution plots

**Biological Interpretation:**
- **Positive anomalies (enrichment)**: More paths than expected given degree structure
  - Indicates biological signal beyond random degree structure
  - May represent functional relationships, regulatory mechanisms, or disease associations
  
- **Novel discoveries (Low DWPC + High Z-score)**: 
  - High anomaly score indicates strong enrichment
  - Low DWPC indicates not explained by simple degree weighting
  - These are the MOST INTERESTING findings for follow-up research
  - Represent context-specific biology not captured by degree alone

**Quadrant Interpretation:**
1. **High DWPC + High Z-score**: Validated enrichments (both methods agree)
2. **High DWPC + Lower Z-score**: Degree-driven enrichments
3. **Lower DWPC + High Z-score**: Novel discoveries (requires further investigation)
4. **Lower DWPC + Lower Z-score**: Modest enrichments

**Next Steps:**
1. Functional enrichment analysis of novel discoveries
2. Literature validation of top enrichments
3. Network visualization of high-confidence enriched connections
4. Comparison across different metapaths
5. Investigate specific compounds/pathways in "novel discoveries" category